# Quiz 04: Web APIs and JSON

This quiz covers web APIs and JSON, from lectures 18 and 19.

- You have 75 minutes. The quiz is open-book and open-notes
- Fork and clone this repository, and work in your clone
- The data files you need are already in the `data/` folder
- Run the cells from top to bottom, and run them all again before you submit
- Tasks 4, 6, 7, 8, 9 and 10 send real requests to live APIs
- Tasks 7 and 10 run `curl` in the terminal. Paste the commands and their output into the answer cells

Write your answers in the empty cells below each task. The bonus task is optional.


In [ ]:
import json

import requests
import pandas as pd


Setting up:
```bash
gh auth status

cd ~/Desktop/"2026 Fall"/"DATASCI 350"/Quizzes
git clone https://github.com/James-04-12/datasci350-quiz04.git
cd datasci350-quiz04
pwd
ls -a
ls data scripts

source ~/Desktop/"2026 Fall"/"DATASCI 350"/.venv/bin/activate
pip install requests pandas pyarrow
```

## Task 1

Look at this URL:

```
https://api.worldbank.org/v2/country/URY/indicator/SP.POP.TOTL?format=json
```

Write the scheme, the host, the path and the query string, one on each line. Then write the full URL for France's population from 2015 to 2018.

**Your answer:**

Lecture 18
```bash
- Scheme: `https`
- Host: `api.worldbank.org`
- Path: `/v2/country/URY/indicator/SP.POP.TOTL`
- Query string: `format=json`
```

France's population, 2015 to 2018:

`https://api.worldbank.org/v2/country/FRA/indicator/SP.POP.TOTL?format=json&date=2015:2018`


## Task 2

Load `data/shuttle.json` with `json.load` into a variable called `reply`. Print:

1. the name of the stop
2. how many arrivals the reply lists
3. the wait for the second arrival, with its unit, for example `5 minutes`


In [3]:
# Task 2

with open("data/shuttle.json") as f: #opening file L19
    reply = json.load(f)

#Emory Point
print(reply["stop"]["name"])

#using len to count number of items (lecutre 18)
print(len(reply["arrivals"]))

print(reply["arrivals"][1]["wait"], reply["units"]["wait"])

Emory Point
3
11 minutes


## Task 3

Load `data/wb_electricity_ken.json` into a variable called `wb`. It holds the share of Kenya's (KEN) population with access to electricity. Like many APIs, the World Bank sends a list with two elements: metadata first, then the records. Print the number of records the metadata says there are, and check it against the length of the records list. Then print the value for 2020.


In [ ]:
# Task 3

with open("data/wb_electricity_ken.json") as f:
    wb = json.load(f)

#note to self L19, position 0 is metadata and position 1 is list of records
meta = wb[0]
records = wb[1]

print(meta["total"]) #how many they say there are
print(len(records)) #the actual length
print(meta["total"] == len(records)) #check for number validity

print(records[4]["date"], records[4]["value"])

#Lecture 19

13
13
True
2020 71.5


## Task 4

Ask [Open-Meteo](https://open-meteo.com/en/docs) (`https://api.open-meteo.com/v1/forecast`) for the current wind speed at Emory: latitude 33.79, longitude -84.32, and `wind_speed_10m` as the `current` variable in the query. Pass the three values as a dict to `params`, with `timeout=10`, and save the reply as `r`. Print `r.url` and `r.status_code`. Then print the wind speed with its unit.


In [ ]:
# Task 4

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 33.79, "longitude": -84.32, "current": "wind_speed_10m",
}

r = requests.get(url, params=params, timeout=10)
r.raise_for_status()
print(r.url)
print(r.status_code)

data = r.json()
print(data["current"]["wind_speed_10m"], data["current_units"]["wind_speed_10m"])

https://api.open-meteo.com/v1/forecast?latitude=33.79&longitude=-84.32&current=wind_speed_10m
200
8.1 km/h


## Task 5

Without running any code, write the full URL that this call sends. Some characters must be percent-encoded in a URL. Encode them as `requests` would (lecture 18).

```python
requests.get("https://api.example.com/v2/books",
             params={"author": "Toni Morrison", "year": "1970:1990", "format": "json"})
```

**Your answer:**

From lecture 18, we know that the space in "Toni Morrison" = "+"; and for ":" in "1970:1990" it becomes "%3A". with the request adding "?" before the first option and "&" between the others. That said, we would end with having: 

```bash
https://api.example.com/v2/books?author=Toni+Morrison&year=1970%3A1990&format=json
```


## Task 6

Make two bad requests and print the status code and the body (`r.json()`) of each:

1. The Task 4 request with latitude 9999
2. `https://api.worldbank.org/v2/country/KEN/indicator/EG.ELC.ACCS.ZZ?format=json` (the indicator is misspelled)

For each, say in one sentence whether `raise_for_status()` would stop the script. Then write one line of code that would catch the second error.


In [8]:
# Task 6

#lattitude 9999
params_bad = {
    "latitude": 9999, "longitude": -84.32, "current": "wind_speed_10m",
}
r1 = requests.get("https://api.open-meteo.com/v1/forecast", params=params_bad, timeout=10)
print(r1.status_code)
print(r1.json())

#spelling
r2 = requests.get("https://api.worldbank.org/v2/country/KEN/indicator/EG.ELC.ACCS.ZZ?format=json", timeout=10)
print(r2.status_code)
print(r2.json())

#second error
payload = r2.json()
if len(payload) < 2: print("No data:", payload)

400
{'reason': 'Latitude must be in range of -90 to 90°. Given: 9999.0.', 'error': True}
200
[{'message': [{'id': '120', 'key': 'Invalid value', 'value': 'The provided parameter value is not valid'}]}]
No data: [{'message': [{'id': '120', 'key': 'Invalid value', 'value': 'The provided parameter value is not valid'}]}]


**Your answer:**

For open-meteo, they sent back status 400, which means that this is a bad request. That said, "raise_for_status" would stop the script. We know this since it raises an error for whenever we have 4xx or 5xx.

And in terms of the world bank, it sent back 200 even though it was showing that the indicator doesn't exist. This would mean that "raise_for_status()" would've not stopped the script. The error is hidden inside JSON, which only only on item/error message instead of two.

## Task 7

In the terminal, use `curl` to ask [Open-Meteo](https://open-meteo.com/en/docs) (`https://api.open-meteo.com/v1/forecast`) for the weather in Montevideo: latitude -34.90, longitude -56.19, and `relative_humidity_2m` as the `current` variable. Paste into the answer cell, as text:

1. The `curl` command that prints only the status line, and its output
2. The `curl` command that saves the reply as `montevideo.json`, in the top folder of the repository
3. The Python command that prints the saved file indented, and the humidity with its unit


**Your answer:**

```bash
BASE="https://api.open-meteo.com/v1/forecast"
Q="latitude=-34.90&longitude=-56.19&current=relative_humidity_2m"
curl -si "$BASE?$Q" | head -n 1
curl -s -o montevideo.json "$BASE?$Q"
python3 -m json.tool montevideo.json
```

Output:
```bash
(.venv) (base) jamestsai@Jamess-MacBook-Pro-1098 datasci350-quiz04 % BASE="https://api.open-meteo.com/v1/forecast"
(.venv) (base) jamestsai@Jamess-MacBook-Pro-1098 datasci350-quiz04 % Q="latitude=-34.90&longitude=-56.19&current=relative_humidity_2m"
(.venv) (base) jamestsai@Jamess-MacBook-Pro-1098 datasci350-quiz04 % curl -si "$BASE?$Q" | head -n 1
HTTP/1.1 200 OK
(.venv) (base) jamestsai@Jamess-MacBook-Pro-1098 datasci350-quiz04 % curl -s -o montevideo.json "$BASE?$Q"
(.venv) (base) jamestsai@Jamess-MacBook-Pro-1098 datasci350-quiz04 % python3 -m json.tool montevideo.json
{
    "latitude": -34.90334,
    "longitude": -56.1929,
    "generationtime_ms": 0.038743019104003906,
    "utc_offset_seconds": 0,
    "timezone": "GMT",
    "timezone_abbreviation": "GMT",
    "elevation": 7.0,
    "current_units": {
        "time": "iso8601",
        "interval": "seconds",
        "relative_humidity_2m": "%"
    },
    "current": {
        "time": "2026-11-19T23:00",
        "interval": 900,
        "relative_humidity_2m": 80
    }
}
```

In [ ]:
#python input

import json
data = json.load(open("montevideo.json"))
print(data["current"]["relative_humidity_2m"], data["current_units"]["relative_humidity_2m"])

80 %


## Task 8

The script `scripts/pull_data.py` downloads one World Bank indicator and saves two files in `data/raw/`. Change its four settings to collect the share of the population using the internet in Kenya, Ghana and Senegal, from 2015 to 2024, with `internet` as the output name. Find the indicator code at [data.worldbank.org/indicator](https://data.worldbank.org/indicator).

Run the script from the top folder of this repository, and paste the settings you used and the script's full output into the answer cell. Then answer briefly:

1. How many rows did you get, and why does the script save the raw reply as well as the CSV?
2. Look at Ghana's values in `data/raw/internet.csv`. Would you trust the value for 2019? Say why, and what you would do about it in an analysis.


**Your answer:**

things we changed:
```bash
INDICATOR = "IT.NET.USER.ZS"
COUNTRIES = "KEN;GHA;SEN"
YEARS = "2015:2024"
OUTPUT_NAME = "internet"
```

terminal:
```bash
python scripts/pull_data.py
cat data/raw/internet.csv
```
Output:
```bash
(.venv) (base) jamestsai@Jamess-MacBook-Pro-1098 datasci350-quiz04 % python scripts/pull_data.py
/Users/jamestsai/Desktop/2026 Fall/DATASCI 350/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Requesting https://api.worldbank.org/v2/country/KEN;GHA;SEN/indicator/IT.NET.USER.ZS
Saved 30 rows to data/raw/internet.csv
  country_code country  year      value
0          GHA   Ghana  2015  23.000000
1          GHA   Ghana  2016  28.000000
2          GHA   Ghana  2017  37.884201
3          GHA   Ghana  2018  43.000000
4          GHA   Ghana  2019  22.073200
(.venv) (base) jamestsai@Jamess-MacBook-Pro-1098 datasci350-quiz04 % cat data/raw/internet.csv
country_code,country,year,value
GHA,Ghana,2015,23.0
GHA,Ghana,2016,28.0
GHA,Ghana,2017,37.88420105
GHA,Ghana,2018,43.0
GHA,Ghana,2019,22.0732
GHA,Ghana,2020,62.51760101
GHA,Ghana,2021,68.6
GHA,Ghana,2022,69.91329956
GHA,Ghana,2023,70.67430115
GHA,Ghana,2024,72.17569733
KEN,Kenya,2015,16.5878548475992
KEN,Kenya,2016,16.6
KEN,Kenya,2017,17.82710075
KEN,Kenya,2018,19.5
KEN,Kenya,2019,22.6687733618171
KEN,Kenya,2020,24.7227993
KEN,Kenya,2021,26.96290016
KEN,Kenya,2022,29.40589905
KEN,Kenya,2023,32.07040024
KEN,Kenya,2024,34.97622513
SEN,Senegal,2015,21.70000076
SEN,Senegal,2016,25.66480064
SEN,Senegal,2017,29.64310074
SEN,Senegal,2018,35.29999924
SEN,Senegal,2019,43.37969971
SEN,Senegal,2020,53.30879974
SEN,Senegal,2021,55.04040146
SEN,Senegal,2022,56.82830048
SEN,Senegal,2023,59.18740082
SEN,Senegal,2024,60.11780167
(.venv) (base) jamestsai@Jamess-MacBook-Pro-1098 datasci350-quiz04 % 
```

1. From terminal, we got 30 rows (three countries multiplies by 10 years) with no rows dropped for missing values. This is because we get to see the raw output as well, if we need to reorganize, we can directly do it from the raw file without calling for API again (since data came from API straight and untouched).

2. In short, I would not trust it. This is becasue statistically, we know that it drops from 43% in 2018 to 22% in 2019 later jumping back up to a 62.5% in 2020. Internet use doesn't just cut in half and then triple in two years, so it's propbably either outliers or a data error or a change in how things are being measured. For analysis, I would flag it while checking the raw JSON and the World Bank Site for notes for that said year, and decided whether or not to readjust or scratch it.

## Task 9

[Frankfurter](https://frankfurter.dev/v1/) is a free API for exchange rates. Its latest rates are at `https://api.frankfurter.dev/v1/latest`, and it takes two options: `base`, the currency you convert from, and `symbols`, the currencies you want, separated by commas.

Using `requests` and `params`, ask for the latest rates from `USD` to `BRL` and `EUR`. Print `r.url`, the date of the rates, and the BRL rate.

For a range of dates, replace `latest` with `start..end`, for example `2020-01-06..2020-01-19` (two weeks, Monday to Sunday). Request that range with the same `params`. Print how many days of rates you got and the BRL rate on 15 January 2020. In one sentence, say why there are fewer than 14 days.


In [ ]:
# Task 9

#most recent rates
url = "https://api.frankfurter.dev/v1/latest"
params = {"base": "USD", "symbols": "BRL,EUR"}

r = requests.get(url, params=params, timeout=10)
r.raise_for_status()
print(r.url)

#defining date ranges
data = r.json()
print(data["date"])
print(data["rates"]["BRL"])

url_range = "https://api.frankfurter.dev/v1/2020-01-06..2020-01-19"
r2 = requests.get(url_range, params=params, timeout=10)
r2.raise_for_status()

data2 = r2.json()
print(len(data2["rates"]))
print(data2["rates"]["2020-01-15"]["BRL"])

https://api.frankfurter.dev/v1/latest?base=USD&symbols=BRL%2CEUR
2026-11-19
5.892
10
4.1599


**Your answer:**

We had 10 days only because the rates we used came from the European Central Bank; publications only occur on working days for weekends show no rates.


## Task 10

Frankfurter's two options are `base` and `symbols` (Task 9). Run these three commands, paste each reply, and say in one sentence how the API treated the options in each. Then say whether checking the status code alone would have caught any of the problems.

```bash
curl -s "https://api.frankfurter.dev/v1/latest?Base=USD&symbols=BRL"
curl -s "https://api.frankfurter.dev/v1/latest?base=USD&symbol=BRL"
curl -s "https://api.frankfurter.dev/v1/latest?base=usd&symbols=brl"
```

**Your answer:**

Output:
```bash
(.venv) (base) jamestsai@Jamess-MacBook-Pro-1098 datasci350-quiz04 % curl -s "https://api.frankfurter.dev/v1/latest?Base=USD&symbols=BRL"
{"amount":1.0,"base":"EUR","date":"2026-11-19","rates":{"BRL":6.4765}}%                                                                                                
(.venv) (base) jamestsai@Jamess-MacBook-Pro-1098 datasci350-quiz04 % curl -s "https://api.frankfurter.dev/v1/latest?base=USD&symbol=BRL"
{"amount":1.0,"base":"USD","date":"2026-11-19","rates":{"AUD":1.49,"BRL":5.892,"CAD":1.43,"CHF":0.84261,"CNY":6.6667,"CZK":22.223,"DKK":6.7997,"EUR":0.90975,"GBP":0.76038,"HKD":7.8456,"HUF":336.09,"IDR":18261,"ILS":3.2887,"INR":93.3,"ISK":124.82,"JPY":159.97,"KRW":1328.45,"MXN":18.9183,"MYR":4.1125,"NOK":9.6866,"NZD":1.8095,"PHP":63.695,"PLN":4.0184,"RON":4.8647,"SEK":10.141,"SGD":1.2897,"THB":33.905,"TRY":50.468,"ZAR":17.2806}}%                                                                    
(.venv) (base) jamestsai@Jamess-MacBook-Pro-1098 datasci350-quiz04 % curl -s "https://api.frankfurter.dev/v1/latest?base=usd&symbols=brl"
{"amount":1.0,"base":"USD","date":"2026-11-19","rates":{"BRL":5.892}}%  
```

#1. Base was written with a capital B, so it ignored it without telling us and used its default base, eur, which means we got the euro-to-brl rate (6.4765) instead of usd

#2 not too sure, the ouputs all showcases symbols with s, so without s it has difficulty reading

#3. since api accepts lowercase values, everything was processed properly this time. Hence, we have usd to brl at 5.892




## Bonus task

Write a `while` loop that reads `data/wb_gdppc_2023_page{n}.json`, starting at `n = 1`, and extends a list called `records`, which starts empty. Stop using the `pages` field from the metadata. Do not hard-code 3. Print `len(records)` at the end: it should match the `total` field, which is 265.

In [ ]:
# Bonus task

When you are done, run all the cells from the top one more time and save the notebook. Then stage, commit and push everything to your fork: `quiz04.ipynb`, `montevideo.json`, your edited `scripts/pull_data.py` and the two files in `data/raw/`. Post your fork's link on Canvas.

Best of luck!

AI Reference:

From task 1 to 9, I used lecture slides only. Specifically the appendix to guide me through. I was having trouble with Task 10, and I asked Gemini to quickly explain to me how to properly read url bases as well as the terminal outputs. I used what I learend in Internatinal Finance for reference as well since we were learning exchange rates.

Thank you again for helping me out so much for the past few weeks.
